# Python-аналитика

# Анализ A/B-теста по SMS-напоминаниям

## Задание

#### Контекст

Команда протестировала новый текст SMS-напоминания. Нужно понять, дал ли он эффект.

#### Дан файл

`sms_ab_test.csv`

* `customer_id`
* `group_name` - `control` / `treatment`
* `segment` - сегмент клиента
* `delivered_flg` - SMS доставлено или нет (`0/1`)
* `paid_7d_flg` - была ли оплата в течение 7 дней (`0/1`)
* `paid_amount_7d` - сумма оплаты в течение 7 дней

#### Что нужно сделать

Напишите код, который:

* считает по каждому `segment` и в целом:
  * количество клиентов;
  * долю доставленных SMS;
  * `payment_rate_7d`;
  * `avg_paid_amount_per_customer_7d`;
* считает uplift `treatment` относительно `control` по `payment_rate_7d`;
* для общего `payment_rate_7d` проводит проверку статистической значимости разницы между `control` и `treatment`;
* формирует короткий вывод на 3-5 предложений.

#### Ограничения

* При расчете `payment_rate_7d` используйте всех клиентов из выборки, а не только доставленные SMS.
* Если используете статистический тест, явно напишите, почему выбрали именно его.

#### Что прислать

Код, итоговые таблицы и краткий вывод.

## Решение

### Постановка A/B-теста

В эксперименте сравниваются две группы:
- `control` (группа A) — пользователи с текущим текстом SMS;
- `treatment` (группа B) — пользователи с новым текстом SMS.

**Целевая метрика:** `payment_rate_7d` — доля клиентов, совершивших оплату в течение 7 дней.

1. **Нулевая гипотеза (H0):** разницы в `payment_rate_7d` между группами нет.  
2. **Альтернативная гипотеза (H1):** разница в `payment_rate_7d` между группами есть.

Для проверки статистической значимости будет использован `z-тест` для сравнения долей.

### Импорт и загрузка

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
df = pd.read_csv('sms_ab_test.csv')

### Просмотр фрейма

In [ ]:
df.head()

,customer_id,group_name,segment,delivered_flg,paid_7d_flg,paid_amount_7d
0,300000,control,high_risk,1,0,0.0
1,300001,treatment,medium_risk,1,0,0.0
2,300002,control,medium_risk,1,0,0.0
3,300003,treatment,high_risk,1,0,0.0
4,300004,control,medium_risk,1,0,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     7200 non-null   int64  
 1   group_name      7200 non-null   object 
 2   segment         7200 non-null   object 
 3   delivered_flg   7200 non-null   int64  
 4   paid_7d_flg     7200 non-null   int64  
 5   paid_amount_7d  7200 non-null   float64
dtypes: float64(1), int64(3), object(2)
memory usage: 337.6+ KB


In [ ]:
df.shape

(7200, 6)

In [ ]:
df['group_name'].value_counts(dropna=False) # баланс

,count
group_name,
control,3600
treatment,3600


Группы **сбалансированы**

In [ ]:
df['segment'].value_counts(dropna=False) # распределение сегментов

,count
segment,
medium_risk,2620
low_risk,1974
high_risk,1717
newcomers,889


In [ ]:
df[['delivered_flg', 'paid_7d_flg']].apply(pd.Series.value_counts, dropna=False)

,delivered_flg,paid_7d_flg
0,508,5516
1,6692,1684


### Метрики по segment и group_name

In [ ]:
segment_metrics = (df
    .groupby(['segment', 'group_name'], as_index=False)
    .agg(customers_cnt=('customer_id', 'nunique'),
         delivered_rate=('delivered_flg', 'mean'),
         payment_rate_7d=('paid_7d_flg', 'mean'),
         avg_paid_amount_per_customer_7d=('paid_amount_7d', 'mean')))

In [ ]:
segment_metrics

,segment,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,high_risk,control,863,0.900348,0.123986,1234.474762
1,high_risk,treatment,854,0.879391,0.148712,1424.542541
2,low_risk,control,987,0.959473,0.313070,1763.592665
3,low_risk,treatment,987,0.954407,0.372847,2083.512199
4,medium_risk,control,1296,0.932099,0.221451,1623.122955
5,medium_risk,treatment,1324,0.939577,0.237915,1767.628595
6,newcomers,control,454,0.916300,0.180617,1191.874802
7,newcomers,treatment,435,0.935632,0.204598,1296.637839


### Метрики "в целом"

In [ ]:
overall_metrics = (df
    .groupby('group_name', as_index=False)
    .agg(
        customers_cnt=('customer_id', 'nunique'),
        delivered_rate=('delivered_flg', 'mean'),
        payment_rate_7d=('paid_7d_flg', 'mean'),
        avg_paid_amount_per_customer_7d=('paid_amount_7d', 'mean')))

In [ ]:
overall_metrics

,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,control,3600,0.930000,0.218056,1514.082275
1,treatment,3600,0.928889,0.249722,1715.934331


### Uplift(эффект)

In [ ]:
control_rate = overall_metrics.loc[overall_metrics['group_name'] == 'control', 'payment_rate_7d'].values[0]

treatment_rate = overall_metrics.loc[overall_metrics['group_name'] == 'treatment', 'payment_rate_7d'].values[0]

In [ ]:
uplift = treatment_rate - control_rate

In [ ]:
uplift

np.float64(0.031666666666666676)

**Наблюдение:** `+3.17` процентных пункта к оплате — положительный эффект `treatment` группы

In [ ]:
# относительный (Заметка)
# rel_uplift = uplift / control_rate
# относительно baseline

### Статистический тест (проверка значимости)

Для проверки статистической значимости разницы в `payment_rate_7d` использован `z-test` для сравнения долей, так как **метрика является бинарной** (paid_7d_flg: 0/1), а группы control и treatment независимы.

**Дополнительно проведён** `chi-test` для таблицы сопряжённости 2x2 как альтернативный метод проверки. Оба теста подходят для анализа различий долей и при достаточном размере выборки дают согласованные результаты. Chi-test тоже подходит **для бинарной метрики**.

- `z-test` напрямую сравнивает доли между двумя группами
- `chi-test` проверяет зависимость между группой и фактом оплаты в таблице сопряжённости.

In [ ]:
# количество успехов (оплат)
success_control = df.loc[df['group_name'] == 'control', 'paid_7d_flg'].sum()
success_treatment = df.loc[df['group_name'] == 'treatment', 'paid_7d_flg'].sum()

# количество наблюдений
n_control = df.loc[df['group_name'] == 'control', 'paid_7d_flg'].count()
n_treatment = df.loc[df['group_name'] == 'treatment', 'paid_7d_flg'].count()

**Z-test**

In [ ]:
# доли
p_control = success_control / n_control
p_treatment = success_treatment / n_treatment

# объединённая оценка (pooled)
p_pool = (success_control + success_treatment) / (n_control + n_treatment)

# стандартная ошибка
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))

# z-статистика
z_stat = (p_treatment - p_control) / se

# p-value (двусторонний тест)
from scipy.stats import norm
p_value_z = 2 * (1 - norm.cdf(abs(z_stat)))

In [ ]:
z_stat, p_value_z

(np.float64(3.1738611784163986), np.float64(0.0015042561770188811))

**Chi-test**

In [ ]:
from scipy.stats import chi2_contingency

# таблица сопряжённости
contingency_table = [[success_control, n_control - success_control],
                     [success_treatment, n_treatment - success_treatment]]

chi2_stat, p_value_chi, dof, expected = chi2_contingency(contingency_table)

In [ ]:
chi2_stat, p_value_chi

(np.float64(9.897443670669132), np.float64(0.0016550852275749991))

In [ ]:
print("Z-test p-value:", p_value_z)
print("Chi-test p-value:", p_value_chi)

Z-test p-value: 0.0015042561770188811
Chi-test p-value: 0.0016550852275749991


- Вывод по обоим тестам совпадает
- `0.0015 << 0.05` — **эффект является статистически значимым**

### Вывод

Эффект является не только статистически значимым, но и практически заметным:
- В группе `treatment` наблюдается **более высокий** `payment_rate_7d` по сравнению с `control` — **(+3.17 п.п.)**, что указывает на **положительный эффект нового текста SMS**.
- Результаты статистических тестов (`z-test` и `chi-test`) показывают, что **различие является статистически значимым** (`p-value < 0.05`), что позволяет отвергнуть нулевую гипотезу об отсутствии эффекта.
- Таким образом, можно сделать вывод, что **новый текст SMS действительно увеличивает вероятность оплаты клиентов в течение 7 дней**.
- **Дополнительно** наблюдается рост среднего платежа на клиента, что усиливает практическую значимость результата.